In [1]:
import os, shutil
if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install monai nibabel scipy -q

import os, json
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from monai.networks.nets import UNet, AttentionUnet, SegResNet
from monai.losses import DiceLoss as MONAIDiceLoss, FocalLoss
from sklearn.model_selection import KFold
from scipy.ndimage import distance_transform_edt
import nibabel as nib
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

PREPROCESSED_ROOT = "/content/drive/MyDrive/MIA/7960856/ISLES-2022_preprocessed"
MASK_ROOT         = "/content/drive/MyDrive/MIA/7960856/ISLES-2022/derivatives"
UNET_DIR          = "/content/drive/MyDrive/MIA/ISLES_RESULTS"       # ← old UNet weights
OUTPUT_DIR        = "/content/drive/MyDrive/MIA/ENSEMBLE_RESULTS"    # ← new models save here
os.makedirs(OUTPUT_DIR, exist_ok=True)
TARGET_SIZE = (128, 128, 128)
print("✅ Setup done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 44.1 MB/s eta 0:00:00


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
Exception ignored in: <function _xla_gc_callback at 0x7f80433b31a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/lib/__init__.py", line 127, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 


Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB
✅ Setup done


In [ ]:
def augment_data(img, mask):
    if np.random.rand() > 0.5:
        flip_axis = np.random.randint(1, 4)
        img  = torch.flip(img,  [flip_axis])
        mask = torch.flip(mask, [flip_axis])
    if np.random.rand() > 0.5:
        img = torch.clamp(img + torch.randn_like(img) * 0.05, 0, 1)
    return img, mask

class ISLESDataset(Dataset):
    def __init__(self, case_ids, augment=False):
        self.case_ids = case_ids
        self.augment  = augment
    def __len__(self): return len(self.case_ids)
    def __getitem__(self, idx):
        cid         = self.case_ids[idx]
        folder_name = f"{cid}_ses-0001_dwi"
        img_path    = os.path.join(PREPROCESSED_ROOT, folder_name, f"{folder_name}_img.nii.gz")
        img = nib.load(img_path).get_fdata()
        img = np.transpose(img, (3,0,1,2)) if img.ndim==4 else np.expand_dims(img, 0)
        img = torch.tensor(img[:3], dtype=torch.float32)
        img = F.interpolate(img.unsqueeze(0), size=TARGET_SIZE,
                            mode="trilinear", align_corners=False).squeeze(0)
        mask_path = os.path.join(MASK_ROOT, cid, "ses-0001", f"{cid}_ses-0001_msk.nii.gz")
        mask = nib.load(mask_path).get_fdata()
        mask = torch.tensor(np.expand_dims(mask, 0), dtype=torch.float32)
        mask = F.interpolate(mask.unsqueeze(0), size=TARGET_SIZE,
                             mode="nearest").squeeze(0)
        if self.augment:
            img, mask = augment_data(img, mask)
        return img, mask, cid

def compute_dice(pred, target, threshold=0.5):
    pred_bin = (pred > threshold).float()
    return 2.0*(pred_bin*target).sum().item() / (pred_bin.sum().item()+target.sum().item()+1e-5)

def compute_f1(pred, target, threshold=0.5):
    pred_bin = (pred > threshold).float()
    tp = (pred_bin * target).sum().item()
    fp = (pred_bin * (1-target)).sum().item()
    fn = ((1-pred_bin) * target).sum().item()
    p  = tp/(tp+fp+1e-5); r = tp/(tp+fn+1e-5)
    return 2*p*r/(p+r+1e-5)

def run_fold(model, train_cases, val_cases, save_path, epochs=30, patience=8):
    dice_loss  = MONAIDiceLoss(to_onehot_y=False, sigmoid=True)
    focal_loss = FocalLoss(alpha=0.5, gamma=2.0)
    optimizer  = Adam(model.parameters(), lr=5e-4)
    scheduler  = CosineAnnealingLR(optimizer, T_max=epochs)
    train_loader = DataLoader(ISLESDataset(train_cases, augment=True),
                              batch_size=2, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(ISLESDataset(val_cases),
                              batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
    best_dice, patience_counter = 0, 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for imgs, masks, _ in tqdm(train_loader, desc=f"Ep {epoch+1}/{epochs}", leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = 0.7*dice_loss(out, masks) + 0.3*focal_loss(out, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        model.eval()
        val_dice, val_f1 = 0, 0
        with torch.no_grad():
            for imgs, masks, _ in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                preds = model(imgs)
                val_dice += compute_dice(preds, masks)
                val_f1   += compute_f1(preds, masks)
        val_dice /= len(val_loader)
        val_f1   /= len(val_loader)
        print(f"Ep {epoch+1:3d} | Loss {total_loss/len(train_loader):.4f} | Val Dice {val_dice:.4f} | F1 {val_f1:.4f}")
        if val_dice > best_dice:
            best_dice, patience_counter = val_dice, 0
            torch.save(model.state_dict(), save_path)
            print(f"  ✅ Saved (Dice: {best_dice:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  ⏹ Early stop at epoch {epoch+1}"); break
    return best_dice

# splits
all_cases = np.array(sorted([
    f.replace("_ses-0001_dwi","")
    for f in os.listdir(PREPROCESSED_ROOT)
    if f.endswith("_ses-0001_dwi")
]))
kf     = KFold(n_splits=5, shuffle=True, random_state=42)
splits = list(kf.split(all_cases))

def build_attunet():
    return AttentionUnet(spatial_dims=3, in_channels=3, out_channels=1,
                         channels=(32,64,128,256), strides=(2,2,2)).to(device)

def build_segresnet():
    return SegResNet(spatial_dims=3, in_channels=3, out_channels=1,
                     init_filters=32).to(device)

def build_unet():
    return UNet(spatial_dims=3, in_channels=3, out_channels=1,
                channels=(32,64,128,256), strides=(2,2,2), num_res_units=2).to(device)

print(f"✅ {len(all_cases)} cases ready")

✅ 250 cases ready


In [ ]:
print("="*50 + "\n  AttUNet — Fold 1\n" + "="*50)
tr_idx, val_idx = splits[0]
att_f1_dice = run_fold(
    model       = build_attunet(),
    train_cases = list(all_cases[tr_idx]),
    val_cases   = list(all_cases[val_idx]),
    save_path   = os.path.join(OUTPUT_DIR, "attunet_fold1_best.pt")
)
print(f"\n✅ AttUNet Fold 1 Best Dice: {att_f1_dice:.4f}")

  AttUNet — Fold 1


Ep   1 | Loss 0.7007 | Val Dice 0.0861 | F1 0.0861
  ✅ Saved (Dice: 0.0861)


Ep   2 | Loss 0.6930 | Val Dice 0.1761 | F1 0.1761
  ✅ Saved (Dice: 0.1761)


Ep   3 | Loss 0.6877 | Val Dice 0.1368 | F1 0.1368


Ep   4 | Loss 0.6803 | Val Dice 0.1654 | F1 0.1654


...

Ep  26 | Loss 0.3362 | Val Dice 0.4471 | F1 0.4471


Ep  27 | Loss 0.3314 | Val Dice 0.4536 | F1 0.4536
  ✅ Saved (Dice: 0.4536)


Ep  28 | Loss 0.3343 | Val Dice 0.4423 | F1 0.4423


Ep  29 | Loss 0.3259 | Val Dice 0.4554 | F1 0.4554
  ✅ Saved (Dice: 0.4554)


Ep  30 | Loss 0.3276 | Val Dice 0.4534 | F1 0.4534

✅ AttUNet Fold 1 Best Dice: 0.4554


In [ ]:
print("="*50 + "\n  AttUNet — Fold 2\n" + "="*50)
tr_idx, val_idx = splits[1]
att_f2_dice = run_fold(
    model       = build_attunet(),
    train_cases = list(all_cases[tr_idx]),
    val_cases   = list(all_cases[val_idx]),
    save_path   = os.path.join(OUTPUT_DIR, "attunet_fold2_best.pt")
)
print(f"\n✅ AttUNet Fold 2 Best Dice: {att_f2_dice:.4f}")

  AttUNet — Fold 2


Ep   1 | Loss 0.7148 | Val Dice 0.0384 | F1 0.0384
  ✅ Saved (Dice: 0.0384)


Ep   2 | Loss 0.7035 | Val Dice 0.2550 | F1 0.2550
  ✅ Saved (Dice: 0.2550)


Ep   3 | Loss 0.6975 | Val Dice 0.1924 | F1 0.1924


Ep   4 | Loss 0.6924 | Val Dice 0.1758 | F1 0.1757


...

Ep  18 | Loss 0.4319 | Val Dice 0.4562 | F1 0.4562


Ep  19 | Loss 0.4218 | Val Dice 0.4786 | F1 0.4786


Ep  20 | Loss 0.4191 | Val Dice 0.4811 | F1 0.4811


Ep  21 | Loss 0.4110 | Val Dice 0.4984 | F1 0.4984
  ✅ Saved (Dice: 0.4984)


Ep 22/30:  31%|███       | 31/100 [01:13<02:39,  2.31s/it]

In [ ]:
print("="*50 + "\n  AttUNet — Fold 3\n" + "="*50)
tr_idx, val_idx = splits[2]
att_f3_dice = run_fold(
    model       = build_attunet(),
    train_cases = list(all_cases[tr_idx]),
    val_cases   = list(all_cases[val_idx]),
    save_path   = os.path.join(OUTPUT_DIR, "attunet_fold3_best.pt")
)
print(f"\n✅ AttUNet Fold 3 Best Dice: {att_f3_dice:.4f}")
print(f"\n🏁 AttUNet Mean Dice: {np.mean([att_f1_dice, att_f2_dice, att_f3_dice]):.4f}")

  AttUNet — Fold 3


Ep   1 | Loss 0.7133 | Val Dice 0.2430 | F1 0.2430
  ✅ Saved (Dice: 0.2430)


Ep   2 | Loss 0.7022 | Val Dice 0.2488 | F1 0.2488
  ✅ Saved (Dice: 0.2488)


Ep   3 | Loss 0.6962 | Val Dice 0.2447 | F1 0.2447


Ep   4 | Loss 0.6918 | Val Dice 0.2779 | F1 0.2779
  ✅ Saved (Dice: 0.2779)


...

Ep  10 | Loss 0.5612 | Val Dice 0.3581 | F1 0.3581
  ✅ Saved (Dice: 0.3581)


Ep  11 | Loss 0.5258 | Val Dice 0.3776 | F1 0.3776
  ✅ Saved (Dice: 0.3776)


Ep  12 | Loss 0.4990 | Val Dice 0.4474 | F1 0.4474
  ✅ Saved (Dice: 0.4474)


Ep  13 | Loss 0.4756 | Val Dice 0.4557 | F1 0.4557
  ✅ Saved (Dice: 0.4557)


Ep 14/30:  79%|███████▉  | 79/100 [03:29<00:55,  2.64s/it]

In [3]:
import numpy as np

att_fold1 = 0.4554
att_fold2 = 0.4984
att_fold3 = 0.4557

att_dice = {
    1: att_fold1,
    2: att_fold2,
    3: att_fold3,
}

best_fold = max(att_dice, key=att_dice.get)
best_dice = att_dice[best_fold]
mean_dice = np.mean(list(att_dice.values()))
std_dice  = np.std(list(att_dice.values()))

print("="*50)
print("  AttUNet 3-Fold Cross-Validation Summary")
print("="*50)
for f, d in att_dice.items():
    print(f"  Fold {f}: Best Dice = {d:.4f}")
print("-"*50)
print(f"  Best fold     : Fold {best_fold} (Dice = {best_dice:.4f})")
print(f"  Mean Dice     : {mean_dice:.4f} ± {std_dice:.4f}")
print("="*50)

  AttUNet 3-Fold Cross-Validation Summary
  Fold 1: Best Dice = 0.4554
  Fold 2: Best Dice = 0.4984
  Fold 3: Best Dice = 0.4557
--------------------------------------------------
  Best fold     : Fold 2 (Dice = 0.4984)
  Mean Dice     : 0.4698 ± 0.0202
